In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp, lag, when, sqrt, hour, dayofweek
)
from pyspark.sql.window import Window
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline

In [2]:
# ---------------------------------------------
# 1. CONFIG
# ---------------------------------------------
DATA_PATH = "data/morocco_full_dataset.csv"
MODELS_DIR = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

spark = SparkSession.builder.appName("Training-Delivery-Models").getOrCreate()


In [3]:
# ---------------------------------------------
# 2. LOAD DATA
# ---------------------------------------------
df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)


In [4]:
# ---------------------------------------------
# 3. CLEAN DATA
# ---------------------------------------------
df = df.withColumn("Timestamp", to_timestamp("Timestamp"))

df = df.withColumn("Delay_Time_Num", col("Delay Time").cast("double"))
df = df.withColumn("Cost_Num", col("Cost").cast("double"))

df = df.dropna(subset=["GPS Latitude", "GPS Longitude", "Temperature", 
                       "Humidity", "Speed", "Timestamp"])


In [5]:
# ---------------------------------------------
# 4. FEATURE ENGINEERING
# ---------------------------------------------
w = Window.partitionBy("RFID ID").orderBy("Timestamp")

df = df.withColumn("prev_lat", lag("GPS Latitude").over(w))
df = df.withColumn("prev_lon", lag("GPS Longitude").over(w))

df = df.withColumn(
    "dist_approx",
    sqrt((col("GPS Latitude") - col("prev_lat"))**2 +
         (col("GPS Longitude") - col("prev_lon"))**2)
)

df = df.withColumn("time_diff_secs",
        unix_timestamp("Timestamp") - unix_timestamp(lag("Timestamp").over(w))
)

df = df.withColumn(
    "speed_calc",
    when(col("time_diff_secs") > 0, col("dist_approx") / col("time_diff_secs"))
)

df = df.withColumn("temp_alert", when(col("Temperature") > 40, 1).otherwise(0))
df = df.withColumn("hour_of_day", hour("Timestamp"))
df = df.withColumn("day_of_week", dayofweek("Timestamp"))

df = df.fillna(0)

In [6]:
# ---------------------------------------------
# 5. CATEGORICAL ENCODING
# ---------------------------------------------
cat_cols = ["Delivery Status", "Stage", "Route ID", "Entity", "Risk Factor", "Disruption Type"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

onehots = [
    OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_ohe"])
    for c in cat_cols
]

In [7]:
# ---------------------------------------------
# 6. RANDOM FOREST FEATURES
# ---------------------------------------------
rf_features = [
    "GPS Latitude", "GPS Longitude", "Temperature", "Humidity", "Speed",
    "dist_approx", "speed_calc", "time_diff_secs",
    "temp_alert", "Delay_Time_Num", "Cost_Num", "hour_of_day", "day_of_week"
] + [f"{c}_ohe" for c in cat_cols]

assembler_rf = VectorAssembler(inputCols=rf_features, outputCol="rf_features")

rf = RandomForestClassifier(featuresCol="rf_features",
                            labelCol="Delivery Status_idx",
                            numTrees=50)


In [8]:
# ---------------------------------------------
# 7. ANOMALY FEATURES + SCALER
# ---------------------------------------------
anom_cols = ["GPS Latitude", "GPS Longitude", "Temperature", "Humidity", "Speed"]

assembler_anom = VectorAssembler(inputCols=anom_cols, outputCol="anom_features_raw")
scaler_anom = StandardScaler(inputCol="anom_features_raw",
                             outputCol="features_anom",
                             withMean=True, withStd=True)

# Fit scaler separately so consumer can load it
df_anom_ready = assembler_anom.transform(df)
anom_scaler_model = scaler_anom.fit(df_anom_ready)
anom_scaler_model.write().overwrite().save(os.path.join(MODELS_DIR, "scaler_anom"))


In [9]:
# ---------------------------------------------
# 8. KMEANS FOR ANOMALY
# ---------------------------------------------
df_anom_final = anom_scaler_model.transform(df_anom_ready)

kmeans = KMeans(featuresCol="features_anom", k=5, seed=42)
kmeans_model = kmeans.fit(df_anom_final)
kmeans_model.write().overwrite().save(os.path.join(MODELS_DIR, "kmeans_model"))


In [10]:
# ---------------------------------------------
# 9. RANDOM FOREST PIPELINE
# ---------------------------------------------
pipeline_rf = Pipeline(stages=indexers + onehots + [assembler_rf, rf])

rf_model = pipeline_rf.fit(df)
rf_model.write().overwrite().save(os.path.join(MODELS_DIR, "rf_model"))


In [11]:

print("\n==============================")
print(" ALL MODELS TRAINED SUCCESSFULLY ")
print("==============================\n")

spark.stop()


 ALL MODELS TRAINED SUCCESSFULLY 

